# Lecture 5: Document Loaders — Ingesting Data

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## The Big Picture

Imagine you're building an **AI assistant for a bookstore**. Your data lives everywhere:
- Product catalog in a **CSV** file
- Book summaries in **PDF** files
- Author bios on **websites**
- Customer reviews in **text** files

How do you feed ALL of this into your AI? You need **Document Loaders** — and that's exactly what this lecture is about.

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|
| 1 | Document object | The "universal shipping box" for all data |
| 2 | Text & PDF loaders | Reading books and notes |
| 3 | CSV loader | Reading spreadsheets row by row |
| 4 | Web loader | Grabbing info from websites |
| 5 | Cloud loaders | Connecting to Google Drive, YouTube, etc. |
| 6 | Hands-on exercises | Build it yourself! |

> **Did you know?** LangChain has **200+ document loaders** — and they ALL produce the
> same output format. Learn one, and you basically know them all!

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.  
If you've already installed them, skip ahead!

In [1]:
# Install required packages (run once, then you can skip this cell)
%pip install langchain langchain-community pypdf beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


---

## 1. The Document Object — Your Universal Data Container

### The Analogy

Think of a **shipping company**. Whether you're shipping books, electronics, or clothes,
everything goes into a **standardized box** with a **shipping label**.

LangChain works the same way:

```
  +----------------------------------+
  |        Document Object           |
  +----------------------------------+
  |  page_content = "actual text"    |  <-- The item inside the box
  |  metadata     = {source, page..} |  <-- The shipping label
  +----------------------------------+
```

- **`page_content`** (str) — The actual text extracted from the file  
- **`metadata`** (dict) — Extra info: where it came from, page number, author, etc.

**Why does this matter?** Because every loader outputs the same `Document` box,
you can swap a PDF loader for a web loader and your pipeline keeps working!

In [2]:
from langchain_core.documents import Document

# Let's create a Document manually to see what it looks like
sample_document = Document(
    page_content="LangChain makes it easy to build AI apps.",
    metadata={
        "source": "manual_creation",
        "page": 0,
        "author": "Hope to Skill",
    }
)

# Print what's inside our "box"
print(f"Content: {sample_document.page_content}")
print(f"Metadata: {sample_document.metadata}")
print(f"Type: {type(sample_document)}")

Content: LangChain makes it easy to build AI apps.
Metadata: {'source': 'manual_creation', 'page': 0, 'author': 'Hope to Skill'}
Type: <class 'langchain_core.documents.base.Document'>


In [3]:
# You can access metadata fields individually
# Using .get() is safer than ['key'] — it won't crash if the key is missing
source = sample_document.metadata.get("source", "unknown")
page_number = sample_document.metadata.get("page", -1)
author = sample_document.metadata.get("author", "N/A")

print(f"Source: {source}")
print(f"Page: {page_number}")
print(f"Author: {author}")

Source: manual_creation
Page: 0
Author: Hope to Skill


#### What just happened?

We created a `Document` object by hand with some text and metadata.  
In practice, you'll **never** create these manually — the loaders do it for you.  
But now you know exactly what's inside every document your loaders will produce!

**Key takeaway:** `.get("key", default)` is safer than `["key"]` because it returns  
the default value instead of crashing when a key doesn't exist.

---

## 2. TextLoader — Reading Plain Text Files

The simplest loader. It reads a `.txt` file and wraps it in a `Document`.  
Think of it as: *"just read the whole file as one block of text."*

**Best for:** `.txt` files, log files, markdown files, any plain text.

In [4]:
from langchain_community.document_loaders import TextLoader

# Step 1: Create the loader (tell it which file to read)
text_loader = TextLoader(
    file_path="data/sample.txt",
    encoding="utf-8",
)

# Step 2: Actually load the file — this returns a LIST of Documents
text_documents = text_loader.load()

# Step 3: See what we got
print(f"Documents loaded: {len(text_documents)}")
print(f"\n--- The Text ---")
print(text_documents[0].page_content)
print(f"\n--- The Label (Metadata) ---")
print(text_documents[0].metadata)

Documents loaded: 1

--- The Text ---
Natural Language Processing (NLP) - An Introduction

Natural Language Processing (NLP) is a subfield of artificial intelligence
that focuses on the interaction between computers and humans through
natural language. The ultimate goal of NLP is to enable computers to
understand, interpret, and generate human language in a valuable way.

Key Areas of NLP:
1. Text Classification - Categorizing text into predefined groups
2. Named Entity Recognition (NER) - Identifying entities like names, dates
3. Sentiment Analysis - Determining the emotional tone of text
4. Machine Translation - Translating text between languages
5. Question Answering - Automatically answering questions from text

NLP has become increasingly important with the rise of large language models
(LLMs) like GPT, Claude, and LLaMA. These models have revolutionized how
we interact with text data and build intelligent applications.

This sample file is part of the Hope to Skill NLP course.




#### What just happened?

- `TextLoader` read the entire file and put it in **one** Document
- The metadata only has `source` (the file path) — because a text file has no pages or authors
- Notice `.load()` always returns a **list**, even if there's only 1 document

**Pattern to remember:** Every loader follows the same 2-step dance:  
`loader = SomeLoader(source)` → `docs = loader.load()`

---

## 3. PyPDFLoader — Reading PDFs Page by Page

PDFs are tricky because they can have dozens of pages, images, and formatting.  
`PyPDFLoader` handles this by splitting each **page** into its own `Document`.

**Analogy:** If TextLoader reads a whole notebook at once, PyPDFLoader  
tears out each page and hands them to you one by one.

**Best for:** Books, research papers, reports, slide decks exported as PDF.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF — each page becomes a separate Document
pdf_loader = PyPDFLoader(file_path="data/sample.pdf")
pdf_documents = pdf_loader.load()

print(f"Total pages loaded: {len(pdf_documents)}")
print("=" * 50)

# Loop through the first 3 pages only
# pdf_documents[:3] is called "slicing" — it takes items at index 0, 1, 2
# To see ALL pages, change [:3] to just pdf_documents (remove the slice)
# enumerate() gives us both the index (0, 1, 2) and the document in each iteration
for index, doc in enumerate(pdf_documents[:3]):
    print(f"\n--- Page {index + 1} ---")

    # [:200] shows only the first 200 characters (a preview)
    # To see the FULL content of a page, use: print(doc.page_content)
    print(f"Content (first 200 chars): {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")

Total pages loaded: 3

--- Page 1 ---
Content (first 200 chars): Introduction to NLP
Hope to Skill - NLP Course
Natural Language Processing (NLP) is a branch of artificial
intelligence that helps computers understand, interpret,
and manipulate human language.
Topic...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}

--- Page 2 ---
Content (first 200 chars): Chapter 1: What is NLP?
NLP combines computational linguistics with statistical,
machine learning, and deep learning models.
Key NLP tasks include:
- Sentiment Analysis
- Named Entity Recognition
- Ma...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:0

#### What just happened?

- Unlike `TextLoader` (which gave us 1 document), `PyPDFLoader` gave us **3 documents** — one per page!
- Each document's metadata includes `page` (page number) and `source` (file path)
- Page numbers start from **0**, not 1 (like list indexes in Python)

**Quick comparison:**

| Loader | # of Documents | Why? |
|--------|---------------|------|
| TextLoader | 1 | Entire file = 1 document |
| PyPDFLoader | N | Each page = 1 document |

---

### Wait — Is PyPDFLoader the ONLY Way to Read PDFs?

**Nope!** There are actually several PDF libraries in Python, and LangChain has
a loader for each one. They all do the same job (read PDFs), but they have
different strengths. Think of it like cars — a sedan, SUV, and truck all get
you from A to B, but you pick one based on the road ahead.

| Loader | Underlying Library | Best For | Speed | Accuracy |
|--------|--------------------|----------|-------|----------|
| `PyPDFLoader` | `pypdf` | Simple text PDFs, quick start | Fast | Good |
| `PyPDFium2Loader` | `pypdfium2` | Speed-critical tasks, large batches | Fastest | Good |
| `PyMuPDFLoader` | `PyMuPDF` / `fitz` | Complex PDFs with tables, images, formatting | Fast | Best |
| `PDFPlumberLoader` | `pdfplumber` | PDFs with **tables** you need to extract | Medium | Excellent for tables |
| `UnstructuredPDFLoader` | `unstructured` | Scanned PDFs (OCR), mixed layouts | Slow | Great for messy PDFs |
| `AmazonTextractPDFLoader` | AWS Textract | Scanned docs at scale (cloud-based OCR) | Slow | Excellent (paid) |

### When to Use Which?

```
Is your PDF just plain text?
  └─ YES → PyPDFLoader (simplest, we used this today)

Does it have complex tables?
  └─ YES → PDFPlumberLoader or PyMuPDFLoader

Is it a scanned image (no selectable text)?
  └─ YES → UnstructuredPDFLoader (with OCR) or AmazonTextractPDFLoader

Need maximum speed for 1000+ PDFs?
  └─ YES → PyPDFium2Loader

Working with Arabic, Chinese, Hindi, or other non-Latin scripts?
  └─ YES → PyMuPDFLoader (best Unicode support)
```

> **Key Lesson:** The right loader depends on YOUR data.
> Always ask: *What kind of document is this? What language? Does it have tables?
> Is it scanned?* Then search for the loader that fits.
>
> You can browse all LangChain loaders here:
> [LangChain Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)

---

## 4. CSVLoader — Turning Spreadsheet Rows into Documents

This is where it gets interesting! `CSVLoader` turns **each row** of your CSV  
into a separate `Document`. Column names become part of the content.

**Analogy:** Imagine a filing cabinet. Each drawer (row) becomes its own document,  
with the folder label (column names) included.

**Best for:** Product catalogs, customer lists, inventory data, survey results.

In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# Load our bookstore product catalog
csv_loader = CSVLoader(
    file_path="data/sample_products.csv",
    encoding="utf-8",
)
csv_documents = csv_loader.load()

print(f"Total rows loaded as documents: {len(csv_documents)}")

# Loop through the first 3 rows (products) only
# csv_documents[:3] = slice that takes the first 3 items from the list
# To see ALL rows, remove the [:3] → for index, doc in enumerate(csv_documents):
# enumerate() pairs each item with its position number (0, 1, 2, ...)
for index, doc in enumerate(csv_documents[:3]):
    print(f"\n--- Product {index + 1} ---")
    print(f"Content:\n{doc.page_content}")   # full content of this row (it's short)
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)  # prints a line of 40 dashes as a visual separator

Total rows loaded as documents: 8

--- Product 1 ---
Content:
product_id: 1
product_name: Python Crash Course
category: Book
price: 29.99
description: A hands-on project-based introduction to programming
Metadata: {'source': 'data/sample_products.csv', 'row': 0}
----------------------------------------

--- Product 2 ---
Content:
product_id: 2
product_name: NLP with Transformers
category: Book
price: 49.99
description: Building language applications with Hugging Face
Metadata: {'source': 'data/sample_products.csv', 'row': 1}
----------------------------------------

--- Product 3 ---
Content:
product_id: 3
product_name: Wireless Mouse
category: Electronics
price: 15.99
description: Ergonomic wireless mouse with USB receiver
Metadata: {'source': 'data/sample_products.csv', 'row': 2}
----------------------------------------


#### What just happened?

- Our CSV had 8 rows, so we got **8 Documents** (one per row)
- Each document's `page_content` contains the column names AND values  
  (e.g., `product_name: Python Crash Course`)
- The metadata tells us the `source` file and `row` number

**Notice the difference?** A PDF's metadata has `page`, but a CSV's metadata has `row`.  
Different loaders provide different metadata — always check what you're getting!

---

## 5. WebBaseLoader — Grabbing Data from Websites

What if your data isn't in a file at all — it's on a **website**?  
`WebBaseLoader` uses BeautifulSoup to scrape web pages and turn them into Documents.

**Analogy:** It's like copying text from a web page and pasting it into a document,  
but automated and at scale.

**Best for:** Documentation sites, blog posts, Wikipedia articles, news.  
**Not great for:** JavaScript-heavy sites (SPAs) — the content may not load.

In [7]:
from langchain_community.document_loaders import WebBaseLoader

# Let's load the Wikipedia page about NLP
web_loader = WebBaseLoader(
    web_paths=["https://en.wikipedia.org/wiki/Natural_language_processing"],
)
web_documents = web_loader.load()

print(f"Documents loaded: {len(web_documents)}")
print(f"\n--- Metadata ---")
print(web_documents[0].metadata)

# [:500] slices the string to show only the first 500 characters
# Web pages can have 50,000+ characters — too much to print at once!
# To see the FULL page content, use: print(web_documents[0].page_content)
print(f"\n--- Content (first 500 chars) ---")
print(web_documents[0].page_content[:500])

USER_AGENT environment variable not set, consider setting it to identify your requests.


Documents loaded: 1

--- Metadata ---
{'source': 'https://en.wikipedia.org/wiki/Natural_language_processing', 'title': 'Natural language processing - Wikipedia', 'language': 'en'}

--- Content (first 500 chars) ---




Natural language processing - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





Donate Create account L


In [8]:
# You can load MULTIPLE web pages at once!
multi_loader = WebBaseLoader(
    web_paths=[
        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://en.wikipedia.org/wiki/Artificial_intelligence",
    ],
)
multi_docs = multi_loader.load()

# This loop goes through EVERY document in multi_docs (no slicing needed — only 2 docs)
# enumerate() gives us the index (0, 1) alongside each document
for index, doc in enumerate(multi_docs):
    # .get() safely reads a key from the metadata dictionary
    # If the key doesn't exist, it returns the fallback value ("Unknown" / "No Title")
    source = doc.metadata.get("source", "Unknown")
    title = doc.metadata.get("title", "No Title")

    # len() counts the total number of characters in the page content
    content_length = len(doc.page_content)

    print(f"\nDocument {index + 1}:")
    print(f"  Title: {title}")
    print(f"  Source: {source}")
    # :, inside the f-string adds commas to large numbers (e.g., 85432 → 85,432)
    print(f"  Content Length: {content_length:,} characters")


Document 1:
  Title: Machine learning - Wikipedia
  Source: https://en.wikipedia.org/wiki/Machine_learning
  Content Length: 124,930 characters

Document 2:
  Title: Artificial intelligence - Wikipedia
  Source: https://en.wikipedia.org/wiki/Artificial_intelligence
  Content Length: 209,273 characters


#### What just happened?

- Each URL became **one Document** (the entire page's text)
- Web metadata includes `source` (URL), `title`, and sometimes `description` and `language`
- The content might look messy — that's because web pages have navigation menus,  
  footers, and other text mixed in. We'll learn how to clean this in a later lecture!

**Tip:** `{content_length:,}` adds commas to large numbers (e.g., `85,432` instead of `85432`).  
A nice Python trick for readability!

---

## 6. SitemapLoader — Crawling Entire Websites

Want to load an **entire website**, not just one page?  
`SitemapLoader` reads a site's `sitemap.xml` file (which lists all pages)  
and loads each page as a Document.

**Analogy:** `WebBaseLoader` reads one page of a book. `SitemapLoader` reads  
the **table of contents** and then loads every chapter.

**Best for:** Building a knowledge base from docs sites, company wikis.  
**Warning:** This can be SLOW for large sites — use `filter_urls` to limit scope.

In [17]:
# SitemapLoader — Reference Only (takes too long for class demo)
# Uncomment the code below and run this cell to try it at home!

# IMPORTANT: Jupyter notebooks already run an "event loop" (async engine).
# SitemapLoader uses asyncio internally, which conflicts with Jupyter's loop.
# nest_asyncio patches this so both can work together.
# You MUST run these two lines before using SitemapLoader in a notebook.
# run this once if not installed

# %pip install nest_asyncio  
# %pip install tqdm

# import nest_asyncio
# nest_asyncio.apply()  # allows nested event loops — fixes the RuntimeError

# from langchain_community.document_loaders.sitemap import SitemapLoader

# # filter_urls limits which pages to load (otherwise it loads EVERYTHING)
# sitemap_loader = SitemapLoader(
#     web_path="https://docs.python.org/3/sitemap.xml",
#     filter_urls=["https://docs.python.org/3/tutorial"],
# )

# sitemap_docs = sitemap_loader.load()
# print(f"Total pages loaded: {len(sitemap_docs)}")

print("SitemapLoader is commented out to save time in class.")
print("Try it at home — remember to uncomment nest_asyncio lines first!")

SitemapLoader is commented out to save time in class.
Try it at home — remember to uncomment nest_asyncio lines first!


---

## 7. Cloud & Special Loaders — The Full Menu

LangChain isn't limited to local files and web pages. It can connect to  
cloud services and specialized platforms too:

| Loader | What It Reads | You'll Need |
|--------|--------------|-------------|
| `GoogleDriveLoader` | Google Docs, Sheets | OAuth credentials |
| `NotionLoader` | Notion pages & databases | Notion API token |
| `YoutubeLoader` | Video transcripts (captions) | `youtube-transcript-api` |
| `S3DirectoryLoader` | Files from AWS S3 buckets | AWS credentials |

**Why is this powerful?** Imagine building a chatbot that can answer questions  
about your company — it could pull data from your Google Drive docs, Notion wiki,  
YouTube training videos, and S3 file storage all at once!

In [18]:
# These loaders need API keys/credentials to work.
# Here's how you'd USE them — save this as a reference for your projects!

# --- YouTube Transcript Loader ---
# Great for: turning tutorial videos into searchable text
#
# from langchain_community.document_loaders import YoutubeLoader
#
# loader = YoutubeLoader.from_youtube_url(
#     url="https://www.youtube.com/watch?v=VIDEO_ID",
#     add_video_info=True,
# )
# docs = loader.load()

# --- Google Drive Loader ---
# Great for: loading team documents, shared reports
#
# from langchain_community.document_loaders import GoogleDriveLoader
#
# loader = GoogleDriveLoader(
#     folder_id="your-google-drive-folder-id",
#     credentials_path="credentials.json",
# )
# docs = loader.load()

# --- Notion Loader ---
# Great for: importing your team's knowledge base
#
# from langchain_community.document_loaders import NotionDirectoryLoader
#
# loader = NotionDirectoryLoader(path="Notion_DB_Export")
# docs = loader.load()

print("These loaders require API keys — see comments above for usage.")
print("The pattern is always the same: create loader -> call .load()")

These loaders require API keys — see comments above for usage.
The pattern is always the same: create loader -> call .load()


---

## 8. Hands-On Part 1: The PDF Explorer

Time to put it all together! Let's write a reusable function that loads  
any PDF and gives us a nice summary. This is something you'll actually  
use in real projects.

In [11]:
from langchain_community.document_loaders import PyPDFLoader


def explore_pdf(file_path):
    """Load a PDF and print a nice summary of its contents."""

    # Load every page as a separate Document
    loader = PyPDFLoader(file_path=file_path)
    documents = loader.load()

    # Summary header
    print(f"PDF: {file_path}")
    print(f"Total pages: {len(documents)}")
    print("=" * 60)  # prints 60 equal signs as a visual divider

    # min(3, len(documents)) picks the SMALLER value
    # If the PDF has 50 pages, we show 3. If it has 2 pages, we show 2.
    # This prevents crashing when the PDF has fewer than 3 pages.
    pages_to_show = min(3, len(documents))

    # range(pages_to_show) generates numbers: 0, 1, 2 (up to pages_to_show - 1)
    # We use these as indexes to access each document from the list
    for i in range(pages_to_show):
        doc = documents[i]
        print(f"\n--- Page {i + 1} ---")

        # [:300] shows only the first 300 characters as a preview
        # To see the FULL page, replace with: print(doc.page_content)
        print(f"Content preview: {doc.page_content[:300]}")
        print(f"Metadata: {doc.metadata}")
        print("-" * 60)

    # Show what metadata keys are available
    if documents:
        # .keys() returns all the key names from the metadata dictionary
        # list() converts it into a readable list format
        print(f"\nAvailable metadata keys: {list(documents[0].metadata.keys())}")

    # Return the documents so they can be used later in your code
    return documents


# Try it! Replace 'data/sample.pdf' with your own PDF path if you have one
pdf_docs = explore_pdf("data/sample.pdf")

PDF: data/sample.pdf
Total pages: 3

--- Page 1 ---
Content preview: Introduction to NLP
Hope to Skill - NLP Course
Natural Language Processing (NLP) is a branch of artificial
intelligence that helps computers understand, interpret,
and manipulate human language.
Topics covered in this course:
1. Text Processing and Tokenization
2. Word Embeddings and Vector Represen
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
------------------------------------------------------------

--- Page 2 ---
Content preview: Chapter 1: What is NLP?
NLP combines computational linguistics with statistical,
machine learning, and deep learning models.
Key NLP tasks include:
- Sentiment Analysis
- Named Ent

#### What just happened?

We built a **reusable function** that works with any PDF. A few things to note:

- `min(3, len(documents))` makes sure we don't crash if the PDF has less than 3 pages
- `documents[0].metadata.keys()` shows us what info the loader provides
- The function **returns** the documents so you can use them later in your pipeline

**Try this:** Call `explore_pdf()` with a PDF from your own computer!

---

## 9. Hands-On Part 2: Loader Showdown — Web vs CSV

Let's load data from **two completely different sources** and see how  
LangChain standardizes them into the same format. This is where the  
"universal shipping box" analogy really clicks!

In [19]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders.csv_loader import CSVLoader

# --- Load from the Web ---
web_loader = WebBaseLoader(
    web_paths=["https://en.wikipedia.org/wiki/Natural_language_processing"],
)
web_docs = web_loader.load()

# --- Load from a CSV ---
csv_loader = CSVLoader(
    file_path="data/sample_products.csv",
    encoding="utf-8",
)
csv_docs = csv_loader.load()

# --- Compare them side by side ---
print("LOADER SHOWDOWN: Web vs CSV")
print("=" * 60)

# :<25 means "left-align and pad to 25 characters wide"
# :>15 means "right-align and pad to 15 characters wide"
# This creates a clean, aligned table in the output
print(f"\n{'Feature':<25} {'Web':>15} {'CSV':>15}")
print(f"{'-' * 25} {'-' * 15} {'-' * 15}")
print(f"{'Documents loaded':<25} {len(web_docs):>15} {len(csv_docs):>15}")

# .keys() gets all metadata field names, list() makes it printable
web_keys = list(web_docs[0].metadata.keys())
csv_keys = list(csv_docs[0].metadata.keys())
print(f"{'Metadata keys':<25} {len(web_keys):>15} {len(csv_keys):>15}")

# This calculates the average content length across all documents
# sum(...) adds up all the lengths, then we divide by the count
# "for d in web_docs" loops through every document to get its length
web_avg = sum(len(d.page_content) for d in web_docs) / len(web_docs)
csv_avg = sum(len(d.page_content) for d in csv_docs) / len(csv_docs)
# :,.0f formats the number with commas and no decimal places (e.g., 85,432)
print(f"{'Avg content length':<25} {web_avg:>13,.0f}ch {csv_avg:>13,.0f}ch")

print(f"\nWeb metadata keys: {web_keys}")
print(f"CSV metadata keys: {csv_keys}")

LOADER SHOWDOWN: Web vs CSV

Feature                               Web             CSV
------------------------- --------------- ---------------
Documents loaded                        1               8
Metadata keys                           3               2
Avg content length               54,892ch           136ch

Web metadata keys: ['source', 'title', 'language']
CSV metadata keys: ['source', 'row']


In [20]:
# Despite coming from COMPLETELY different sources,
# both have the same .page_content and .metadata structure!

# hasattr() checks if an object has a specific attribute (returns True/False)
# We're proving that BOTH web and CSV documents have the same structure
print("--- Web Document (same type!) ---")
print(f"Type: {type(web_docs[0])}")
print(f"Has page_content? {hasattr(web_docs[0], 'page_content')}")
print(f"Has metadata? {hasattr(web_docs[0], 'metadata')}")

print(f"\n--- CSV Document (same type!) ---")
print(f"Type: {type(csv_docs[0])}")
print(f"Has page_content? {hasattr(csv_docs[0], 'page_content')}")
print(f"Has metadata? {hasattr(csv_docs[0], 'metadata')}")

print("\n" + "=" * 60)
print("SAME type, SAME structure, DIFFERENT sources!")
print("This is the power of LangChain's standardized Document object.")

--- Web Document (same type!) ---
Type: <class 'langchain_core.documents.base.Document'>
Has page_content? True
Has metadata? True

--- CSV Document (same type!) ---
Type: <class 'langchain_core.documents.base.Document'>
Has page_content? True
Has metadata? True

SAME type, SAME structure, DIFFERENT sources!
This is the power of LangChain's standardized Document object.


#### What just happened?

We loaded data from a **website** and a **CSV file** — two completely different  
sources — and got back the exact same `Document` type. The only differences are:

- **What's inside** `page_content` (full web page text vs one CSV row)
- **What's inside** `metadata` (URL/title vs file path/row number)

**This is the whole point of Document Loaders:** no matter where your data comes  
from, your downstream code (text splitting, embeddings, search) stays the same.

**Best practice:** Always inspect your loaded data with `print()` before  
feeding it into a pipeline. Quality varies a lot between sources!

---

## 10. Mini Challenges

Try these on your own! Solutions are hidden at the bottom of the notebook.

### Challenge 1: The Text Detective
Create a new `.txt` file about your favorite topic, load it with `TextLoader`,  
and print its content and metadata.

### Challenge 2: Metadata Inspector
Load the sample PDF and write code to print **only** the `source` and `page`  
from each document's metadata (ignore the other keys).

### Challenge 3: The Loader Mixer
Load data from all 3 sources (text, PDF, CSV) into one combined list.  
Print the total number of documents and the source of each one.

> **Hint for Challenge 3:** You can combine lists with `+`  
> e.g., `all_docs = text_docs + pdf_docs + csv_docs`

In [14]:
# ============================================================
# Challenge 1: The Text Detective
# ============================================================
# Step 1: Create a .txt file (or use the one in data/sample.txt)
# Step 2: Load it with TextLoader
# Step 3: Print the content and metadata

# Your code here:


In [15]:
# ============================================================
# Challenge 2: Metadata Inspector
# ============================================================
# Load sample.pdf and print ONLY the source and page number
# from each document's metadata.
#
# Expected output format:
#   Page 1 -> source: data/sample.pdf
#   Page 2 -> source: data/sample.pdf
#   ...

# Your code here:


In [16]:
# ============================================================
# Challenge 3: The Loader Mixer
# ============================================================
# Load from all 3 sources, combine into one list,
# and print a summary showing how many docs came from where.
#
# Hint: all_docs = text_docs + pdf_docs + csv_docs

# Your code here:


---

## 11. Quick Reference: Choosing the Right Loader

### The Golden Rule

> **Don't memorize loaders — learn how to FIND the right one.**
>
> There are 200+ loaders and more get added every month. Instead of memorizing,
> ask yourself these 3 questions:
>
> 1. **What type of file/source is it?** (PDF, CSV, web, cloud, database...)
> 2. **What's special about it?** (Has tables? Scanned? Non-English? JavaScript-heavy?)
> 3. **Search for it:** `langchain [your source type] loader`

### Common Loaders — Your Starting Point

| Your Data Source | Default Loader | When to Look for Alternatives |
|-----------------|---------------|-------------------------------|
| `.txt` file | `TextLoader` | Large files → consider chunking |
| `.pdf` (simple text) | `PyPDFLoader` | Has tables → `PDFPlumberLoader`, Scanned → `UnstructuredPDFLoader` |
| `.pdf` (complex) | `PyMuPDFLoader` | Best overall quality, needs `pip install pymupdf` |
| `.csv` file | `CSVLoader` | Big CSVs → `pandas` + custom loader |
| `.docx` / `.pptx` | `UnstructuredFileLoader` | Needs `pip install unstructured` |
| `.html` file | `UnstructuredHTMLLoader` | Or `BSHTMLLoader` for simpler parsing |
| Web page | `WebBaseLoader` | JavaScript-heavy → `PlaywrightURLLoader` |
| Entire website | `SitemapLoader` | Slow for large sites, use `filter_urls` |
| Google Drive | `GoogleDriveLoader` | Needs OAuth setup |
| YouTube video | `YoutubeLoader` | Needs `youtube-transcript-api` |
| Notion | `NotionDirectoryLoader` | Needs API token or exported data |
| AWS S3 | `S3DirectoryLoader` | Needs AWS credentials |
| JSON file | `JSONLoader` | Use `jq_schema` to extract specific fields |
| Markdown | `UnstructuredMarkdownLoader` | Preserves heading structure |
| Email (.eml) | `UnstructuredEmailLoader` | Extracts subject, body, attachments |

### It's Not Just About File Types!

The same principle applies to **every** category of loader. Just like PDFs have
6+ different loaders, web scraping has multiple options too:

| Problem | Default | Better Alternative |
|---------|---------|--------------------|
| Static web page | `WebBaseLoader` | Works great! |
| JavaScript-rendered page | `WebBaseLoader` (fails) | `PlaywrightURLLoader` or `SeleniumURLLoader` |
| Need specific HTML elements | `WebBaseLoader` (too much text) | `WebBaseLoader` with `bs_kwargs` to filter tags |
| Crawl entire site | `SitemapLoader` | `RecursiveUrlLoader` if no sitemap exists |

**The mindset to develop:** When a loader doesn't give you good results,
don't force it — search for an alternative. LangChain probably has one!

---

## 12. Key Takeaways

1. **Every loader returns standardized `Document` objects** — swap loaders without changing code
2. Every Document has **`page_content`** (the text) and **`metadata`** (the label)
3. **Always inspect your data** after loading — quality and metadata vary by source
4. **The pattern is always the same:**
   ```python
   loader = SomeLoader(source)  # Point to your data
   docs = loader.load()         # Get back a list of Documents
   ```

### Next Lecture

**Lecture 6: Text Splitters** — Your documents are loaded, but they're too big  
for an AI to process at once. Next, we'll learn how to split them intelligently!

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

Throughout this notebook, the code follows Python's official style guide (PEP 8).  
Here's a summary of every rule applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `text_loader`, `pdf_documents`, `explore_pdf()` |
| Constants | `UPPER_CASE` | `TARGET_URL` (section 9) |
| Classes | `PascalCase` | `Document`, `TextLoader` (from LangChain) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|
| E401 | One import per line | `from langchain_community.document_loaders import TextLoader` |
| E402 | Imports at the top of the file (or top of notebook section) | All imports are at the top of each code cell |
| — | Group imports: stdlib → third-party → local | We only use third-party imports in this notebook |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|
| E225 | Spaces around operators | `index + 1` not `index+1` |
| E231 | Space after commas | `min(3, len(documents))` |
| E265 | Block comments start with `# ` (space after #) | `# Load the PDF` |
| E303 | Two blank lines before function definitions | See `explore_pdf()` cell |
| E501 | Max line length of 79 characters | Long strings sliced: `doc.page_content[:300]` |

### Best Practices

| Practice | Why | Example |
|----------|-----|---------|
| f-strings for formatting | Cleaner than `.format()` or `%s` | `f"Pages: {len(docs)}"` |
| `enumerate()` for loops | Avoids manual counter variables | `for index, doc in enumerate(docs)` |
| `.get(key, default)` for dicts | Won't crash if key is missing | `metadata.get("source", "unknown")` |
| Trailing commas in multi-line | Makes diffs cleaner in git | See all multi-line function calls |
| Docstrings for functions | Explains what the function does | `explore_pdf()` has a docstring |
| Guard clauses early | Return early if input is invalid | `if not documents: return` |
| Descriptive variable names | Code reads like English | `pdf_loader` not `pl`, `web_documents` not `wd` |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
pdf_loader = PyPDFLoader(file_path="data/sample.pdf")
total_pages = len(documents)
for index, doc in enumerate(documents):
    print(f"Page {index + 1}: {doc.page_content[:100]}")

# BAD (violates PEP 8)
pdfLoader = PyPDFLoader(file_path="data/sample.pdf")   # camelCase
tp = len(documents)                                      # unclear name
i = 0                                                    # manual counter
for d in documents:                                      # single-letter var
    print("Page "+str(i+1)+": "+d.page_content[:100])   # string concat
    i=i+1                                                # no spaces
```